# Estudo de Ablação — Algoritmo Genético (AG)

Este notebook analisa os resultados gerados por `scripts/run_ga_ablation.py` (`results/ga_ablation/ablation.csv`) para identificar o impacto de cada componente do **Algoritmo Genético** na ponderação de ensembles sob concept drift:
1. **Ablação estrutural (`operators`)**: elitismo (`e=0`, `e=2`, `e=5`), seleção de sobreviventes (`generational` vs `mu+lambda`), remoção de crossover (`no_crossover`) e remoção de mutação (`no_mutation`) × `S1..S6`;
2. **Testes estatísticos** (Friedman + Nemenyi com Diagramas CD e Wilcoxon pareado + Holm + $\hat{A}_{12}$);
3. **Desempenho por base e por tipo de drift** (Recorrente, Novo, Incremental e Real);
4. **Ablação de hiperparâmetros**: `tournament_k`, `blx_alpha`, `pc`, `sigma`, `pop_size` e `hyper_sigma` (em S4).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.stats import a12, a12_label, cd_diagram, friedman_nemenyi, holm, wilcoxon_p

RESULTS_DIR = PROJECT_ROOT / "results" / "ga_ablation"
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

df = pd.read_csv(RESULTS_DIR / "ablation.csv")
op_df = df[df["block"] == "operators"].copy()
param_df = df[df["block"] == "params"].copy()

print(f"Total de execuções: {len(df)} (operators={len(op_df)}, params={len(param_df)})")
print("Datasets:", sorted(df["dataset"].unique()))
print("Sementes:", df["seed"].nunique())
print("Variantes AG:", sorted(op_df["ga_variant"].unique()))

## 1. Heatmap e Ranking: Variante AG × Estratégia Pós-Drift (S1–S6)

In [ ]:
piv_grid = op_df.groupby(["ga_variant", "post_drift"])["macro_f1"].mean().unstack()
piv_grid["Média"] = piv_grid.mean(axis=1)
piv_grid = piv_grid.sort_values("Média", ascending=False)
display(piv_grid.round(4))

fig, ax = plt.subplots(figsize=(9.5, 4.8))
sns.heatmap(piv_grid.drop(columns=["Média"]), annot=True, fmt=".4f", cmap="YlOrRd", ax=ax)
ax.set_title("Macro-F1 Médio: Variante AG × Estratégia Pós-Drift (todas as bases)")
ax.set_xlabel("Estratégia Pós-Drift")
ax.set_ylabel("Variante AG")
fig.tight_layout()
fig.savefig(FIG_DIR / "heatmap_ga_x_postdrift.png", dpi=150)
plt.show()

combos = op_df.groupby(["ga_variant", "post_drift"])[["macro_f1", "n_evals", "gens_to_95", "replay_time_s"]].mean()
combos = combos.sort_values("macro_f1", ascending=False)
print("=== Top 12 Combinações (Variante AG + Pós-Drift) ===")
display(combos.head(12).round(4))

## 2. Testes Estatísticos entre as Variantes do AG (Friedman + Nemenyi e Wilcoxon + Holm + $\hat{A}_{12}$)

In [ ]:
piv_all = op_df.pivot_table(
    index=["dataset", "seed", "post_drift"], columns="ga_variant", values="macro_f1"
).dropna()
fr_all = friedman_nemenyi(piv_all)
print(f"=== 6 Variantes AG (todas S1–S6, N={fr_all['N']} blocos) ===")
print(f"Friedman p = {fr_all['p_value']:.4g} | CD = {fr_all['cd']:.3f}")
display(fr_all["avg_ranks"].to_frame("Rank Médio"))
p1 = FIG_DIR / "cd_ga_variants_all.png"
cd_diagram(fr_all["avg_ranks"], fr_all["cd"], f"Variantes AG — todas S1–S6 (N={fr_all['N']})", str(p1), fr_all["p_value"])
display(Image(filename=str(p1)))

best_post = op_df.groupby("post_drift")["macro_f1"].mean().idxmax()
piv_bp = op_df[op_df["post_drift"] == best_post].pivot_table(
    index=["dataset", "seed"], columns="ga_variant", values="macro_f1"
).dropna()
fr_bp = friedman_nemenyi(piv_bp)
print(f"\n=== 6 Variantes AG em {best_post} (N={fr_bp['N']} blocos) ===")
print(f"Friedman p = {fr_bp['p_value']:.4g} | CD = {fr_bp['cd']:.3f}")
display(fr_bp["avg_ranks"].to_frame("Rank Médio"))
p2 = FIG_DIR / f"cd_ga_variants_{best_post}.png"
cd_diagram(fr_bp["avg_ranks"], fr_bp["cd"], f"Variantes AG em {best_post} (N={fr_bp['N']})", str(p2), fr_bp["p_value"])
display(Image(filename=str(p2)))

In [ ]:
best_var = fr_bp["avg_ranks"].index[0]
others = [c for c in piv_bp.columns if c != best_var]
print(f"Melhor variante AG em {best_post}: {best_var}")

w_rows = []
for ds in sorted(op_df["dataset"].unique()):
    sub = op_df[(op_df["dataset"] == ds) & (op_df["post_drift"] == best_post)].pivot_table(
        index="seed", columns="ga_variant", values="macro_f1"
    ).dropna()
    pvals = [wilcoxon_p(sub[best_var], sub[o]) for o in others]
    padj = holm(pvals)
    for o, p, pa in zip(others, pvals, padj):
        e = a12(sub[best_var], sub[o])
        diff = float(sub[best_var].mean() - sub[o].mean())
        res = "=" if pa >= 0.05 else ("+" if diff > 0 else "−")
        w_rows.append({
            "dataset": ds, "comparação": f"{best_var} vs {o}",
            "Δ macro-F1": diff, "p (Holm)": pa, "Â12": e, "efeito": a12_label(e), "resultado": res,
        })

wdf = pd.DataFrame(w_rows)
wl = wdf.groupby("comparação")["resultado"].apply(
    lambda r: f"{(r == '+').sum()}/{(r == '=').sum()}/{(r == '−').sum()}"
).to_frame("Vitórias / Empates / Derrotas")
display(wl)
display(wdf)

## 3. Desempenho das Variantes AG por Base e por Tipo de Drift

In [ ]:
by_ds = op_df[op_df["post_drift"] == best_post].groupby(["dataset", "ga_variant"])["macro_f1"].mean().unstack()
by_ds.loc["Média Geral"] = by_ds.mean()
display(by_ds.round(4))

GROUPS = {
    "Recorrente (SEA, STAGGER)": ["SEA-Rec", "STAGGER-Rec"],
    "Novo (AGR-Abr, AGR-Grad)": ["AGR-Abr", "AGR-Grad"],
    "Incremental (RBF-Inc)": ["RBF-Inc"],
    "Real (Elec2)": ["Elec2"],
}
g_rows = {}
for gname, dss in GROUPS.items():
    sub_g = op_df[(op_df["post_drift"] == best_post) & (op_df["dataset"].isin(dss))]
    if not sub_g.empty:
        g_rows[gname] = sub_g.groupby("ga_variant")["macro_f1"].mean()
g_df = pd.DataFrame(g_rows).T
print("=== Macro-F1 por Tipo de Drift ===")
display(g_df.round(4))

fig, ax = plt.subplots(figsize=(10.5, 4.5))
g_df.plot(kind="bar", ax=ax)
ax.set_ylabel(f"Macro-F1 médio ({best_post})")
ax.set_xlabel("")
ax.set_title("Variantes AG por Tipo de Concept Drift")
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Variante AG", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ga_by_drift_type.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Ablação de Hiperparâmetros do AG (`tournament_k`, `blx_alpha`, `pc`, `sigma`, `pop_size`, `hyper_sigma`)

In [ ]:
if not param_df.empty:
    factors = ["tournament_k", "blx_alpha", "pc", "sigma", "pop_size", "hyper_sigma"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
    for ax, fac in zip(axes.flat, factors):
        sub_f = param_df[param_df["ablation_factor"] == fac]
        if sub_f.empty:
            ax.axis("off")
            continue
        tab = sub_f.groupby(["ablation_value", "ga_variant"])["macro_f1"].mean().unstack()
        print(f"\n=== Ablação: {fac} ===")
        display(sub_f.groupby(["ga_variant", "ablation_value"])[["macro_f1", "n_evals", "gens_to_95", "div_end"]].mean().round(4))
        tab.plot(kind="bar", ax=ax)
        ax.set_title(f"Ablação de {fac}")
        ax.set_xlabel(fac)
        ax.set_ylabel("Macro-F1 médio")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(title="Sobreviventes", fontsize=8)
    fig.suptitle("Ablação de Hiperparâmetros do Algoritmo Genético", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "ablation_ga_params.png", dpi=150)
    plt.show()
else:
    print("Bloco 'params' não presente em ablation.csv.")